# Time Travel & State Forking [Step 3 - Replay and Branch]

> **MLCourse - Agentic AI - LangGraph**

This notebook demonstrates how to use LangGraph's time travel capabilities
to replay conversations from any checkpoint and fork state to explore
alternative branches of execution.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv("D:/projects/python/MLCourse/03_agentic_ai/.env")

True

In [2]:
groq_key = os.environ.get("GROQ_API_KEY", "")
if groq_key:
    print("GROQ_API_KEY found")
else:
    print("GROQ_API_KEY not set - using ChatOllama (local, no key needed)")

GROQ_API_KEY found


### Core imports for time travel features


In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langchain_ollama import ChatOllama


### State schema with a branch label for tracking which path we took


In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list, add_messages]


### Initialize local LLM


In [ ]:
llm = ChatOllama(model="llama3.1:8b", temperature=0)


### Chatbot node calls the LLM and returns the response


In [ ]:
def chatbot(state: ChatState):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}


### Build the graph


In [ ]:
graph_builder = StateGraph(ChatState)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)


### Use MemorySaver for in-memory checkpointing


In [ ]:
checkpointer = MemorySaver()
graph = graph_builder.compile(checkpointer=checkpointer)


### Visualize the graph


In [ ]:
from IPython.display import Image, display
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Graph visualization unavailable: {e}")
    print("Graph nodes: START -> chatbot -> END")


### Execute a multi-turn conversation to build up checkpoint history


In [ ]:
config = {"configurable": {"thread_id": "travel-thread"}}

# Turn 1
response = graph.invoke(
    {"messages": [("user", "I want to learn Python.")]},
    config=config,
)
print("Turn 1:", response["messages"][-1].content[:100])

# Turn 2
response = graph.invoke(
    {"messages": [("user", "Should I start with web dev or data science?")]},
    config=config,
)
print("Turn 2:", response["messages"][-1].content[:100])

# Turn 3
response = graph.invoke(
    {"messages": [("user", "Tell me more about data science.")]},
    config=config,
)
print("Turn 3:", response["messages"][-1].content[:100])


### List all checkpoints - each one is a snapshot of the full state


In [ ]:
checkpoints = list(graph.get_state_history(config))
print(f"\nTotal checkpoints: {len(checkpoints)}")
for cp in checkpoints:
    msg_count = len(cp.values.get("messages", []))
    cp_id = cp.config["configurable"]["checkpoint_id"][:10]
    print(f"  {cp_id}... ({msg_count} messages)")


### Time Travel - Replay from checkpoint


In [ ]:
# Pick the second checkpoint (after turn 1) to replay from
target_checkpoint = checkpoints[1]
print(f"\nReplaying from checkpoint: {target_checkpoint.config['configurable']['checkpoint_id'][:10]}...")
print(f"Messages at that point: {len(target_checkpoint.values['messages'])}")


### Fork the conversation from the selected checkpoint


In [ ]:
# This creates a new branch with a different follow-up message
fork_config = target_checkpoint.config
response = graph.invoke(
    {"messages": [("user", "Actually, I want to focus on web development instead.")]},
    fork_config,
)
print("Fork response (web dev path):", response["messages"][-1].content[:100])


### Now the original thread continues from where we left off


In [ ]:
# The fork did not affect the original branch
response = graph.invoke(
    {"messages": [("user", "What tools do I need for data science?")]},
    config,
)
print("Original thread continues:", response["messages"][-1].content[:100])


### Inspect both branches - original has more checkpoints than before


In [ ]:
original_checkpoints = list(graph.get_state_history(config))
fork_config_with_next = fork_config.copy()
fork_checkpoints = list(graph.get_state_history(fork_config))

print(f"Original branch checkpoints: {len(original_checkpoints)}")
print(f"Forked branch checkpoints: {len(fork_checkpoints)}")


### Show the forked branch state


In [ ]:
fork_state = graph.get_state(fork_config)
print("\nForked branch messages:")
for msg in fork_state.values["messages"]:
    role = msg.type
    content = msg.content[:80] + "..." if len(msg.content) > 80 else msg.content
    print(f"  [{role}] {content}")


### Demonstrate updating state at a checkpoint (manual fork)


In [ ]:
# This lets you modify the state and continue from there
latest_state = graph.get_state(config)
print(f"Latest state has {len(latest_state.values['messages'])} messages")
print("Time travel capabilities:")
print("  get_state_history() - list all checkpoints for a thread")
print("  get_state(config) - load state at a specific checkpoint")
print("  invoke() with old config - fork from any checkpoint")
print("  update_state() - manually modify state at a checkpoint")
